# 3.3 — Bias–Variance Tradeoff

The bias–variance tradeoff explains why the model with the prettiest training score is not always the model that behaves best tomorrow. We will build the pieces from scratch: empirical risk as an average loss, a complexity cost that guards against brittle flexibility, the decomposition of expected squared error into bias, variance, and irreducible noise, and the validation comparison that decides which setting to carry forward.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build the bias–variance tradeoff one idea at a time. Run each cell in order and read the printed intermediate values — every piece of arithmetic is exposed so the decision score never feels like a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, simulation, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for resampling demos.

### 1. Empirical risk is an average over examples

A learning method starts with losses on the training sample. The lesson's toy losses are 0.213, 0.148, and 0.420. Empirical risk is just their average, but that small averaging move matters: it turns many example-level misses into one comparable training number.

In [ ]:
losses_w = np.array([0.213, 0.148, 0.420])  # three verified per-example losses from the lesson.
print("losses:", losses_w)  # inspect the individual misses before averaging.
print("sum:", round(float(losses_w.sum()), 3))  # 0.781, the numerator of empirical risk.

▶ What you'll see: three losses with total 0.781, so the average will be near one third of that.

In [ ]:
risk_w = float(losses_w.mean())  # empirical risk R_S = average training loss.
print("empirical risk R_S:", round(risk_w, 3))  # 0.260.
assert round(risk_w, 3) == 0.260  # concrete check from the lesson block.

▶ What you'll see: `R_S = 0.260`, the raw fit score before any complexity cost.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["ex1", "ex2", "ex3"], losses_w, color="steelblue")
plt.axhline(risk_w, color="crimson", linestyle="--", label=f"mean={risk_w:.3f}")
plt.title("1: empirical risk averages losses"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: the dashed line is the empirical risk; it summarizes the three bars without hiding their spread.

*Why it's done this way:* ERM optimizes an average because a model is judged across examples, not on its best or worst anecdote. The mean is the scale-preserving summary: if every loss doubles, the risk doubles; if we add more comparable examples, the denominator keeps the number on the same loss scale.

### 2. Training score is not the full decision score

A low training loss can be bought with too much flexibility. The lesson adds a method cost of 0.080, standing for complexity, regularization, operational burden, or any guardrail the selection rule must respect. The score used for selection is therefore `risk + cost`, not risk alone.

In [ ]:
cost_w = 0.080  # method cost from the lesson.
score_w = risk_w + cost_w  # final selection score for this candidate.
print("risk:", round(risk_w, 3), "cost:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(score_w, 3) == 0.340

▶ What you'll see: the raw `0.260` becomes a full decision score of `0.340` after adding cost.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["risk", "cost", "risk+cost"], [risk_w, cost_w, score_w], color=["steelblue", "orange", "seagreen"])
plt.title("2: model selection uses the full score"); plt.ylabel("score units"); plt.show()

▶ What you'll see: the decision bar is taller than the raw-risk bar because flexibility is not free.

*Why it's done this way:* If two models have similar training risk, the simpler or cheaper one often generalizes better because it had fewer ways to chase sample accidents. Adding the cost implements that preference numerically: the formula defines what “better” means before comparing models.

### 3. Bias, variance, and irreducible noise separate three failure modes

For squared error, the expected prediction error at a point decomposes as bias² + variance + noise. Bias is systematic miss of the average prediction, variance is how much predictions wobble across training samples, and noise is uncertainty no model can remove.

In [ ]:
true_y_w = 3.0  # the noiseless target at one x value.
preds_w = np.array([2.2, 2.6, 3.1, 3.5, 3.6])  # predictions from five possible training samples.
mean_pred_w = float(preds_w.mean())  # average prediction over samples.
print("predictions:", preds_w)
print("mean prediction:", round(mean_pred_w, 3))

▶ What you'll see: the model's predictions vary across samples and average to about 3.0.

In [ ]:
bias2_w = (mean_pred_w - true_y_w) ** 2  # squared systematic error.
variance_w = float(np.mean((preds_w - mean_pred_w) ** 2))  # prediction spread across samples.
noise_w = 0.09  # irreducible variance sigma^2.
total_w = bias2_w + variance_w + noise_w
print("bias^2:", round(bias2_w, 3), "variance:", round(variance_w, 3), "noise:", round(noise_w, 3))
print("expected error pieces sum:", round(total_w, 3))
assert round(bias2_w, 3) == 0.000 and round(variance_w, 3) == 0.284 and round(total_w, 3) == 0.374

▶ What you'll see: this toy model has almost no bias but noticeable variance; noise stays even if modeling improves.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["bias²", "variance", "noise"], [bias2_w, variance_w, noise_w], color=["gray", "purple", "orange"])
plt.title("3: expected squared error pieces"); plt.ylabel("error contribution"); plt.show()

▶ What you'll see: variance is the main removable term in this toy decomposition.

*Why it's done this way:* Squared error lets the algebra split average error into a mean-location problem and a spread problem. That separation is useful because different fixes target different terms: more flexible models reduce bias, constraints and averaging reduce variance, and no model removes irreducible noise.

### 4. A tempting alternative must beat the full score by enough

The lesson's more flexible alternative reaches score 0.388. Because lower is better, the baseline score 0.340 wins by an absolute gap of 0.048. The relative gap, 0.124, asks whether the win is large compared with the alternative's own scale.

In [ ]:
alternative_w = 0.388  # decision score of the more flexible alternative.
gap_w = alternative_w - round(score_w, 3)  # positive means the rounded baseline score is lower.
relative_gap_w = gap_w / alternative_w  # scale-aware comparison.
print("baseline score:", round(score_w, 3), "alternative:", alternative_w)
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
assert round(gap_w, 3) == 0.048 and round(relative_gap_w, 3) == 0.124

▶ What you'll see: the baseline is better by 0.048, which is about 12.4% of the alternative score.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["baseline", "flexible alt"], [score_w, alternative_w], color=["seagreen", "indianred"])
plt.title("4: compare full decision scores"); plt.ylabel("lower is better"); plt.show()

▶ What you'll see: the flexible alternative is visibly higher even if it may have had a nicer raw training fit.

*Why it's done this way:* A model comparison is meaningful only after both candidates are on the same score scale. The absolute gap tells how many score units separate them; the relative gap warns whether that difference is large enough to survive resampling noise or practical uncertainty.

### 5. Stabilization trades flexibility for lower future risk

A stabilizing knob — regularization, averaging, pruning, or a smaller hypothesis class — reduces the baseline score by 20% in the lesson. That creates a stabilized score of 0.272, illustrating the practical bargain: give up brittle variation to improve the future-facing score.

In [ ]:
shrink_w = 0.80  # a 20% reduction in the decision score.
stable_w = shrink_w * score_w  # stabilized score.
print("original score:", round(score_w, 3))
print("stabilized score:", round(stable_w, 3))
assert round(stable_w, 3) == 0.272

▶ What you'll see: stabilization lowers the score from 0.340 to 0.272.

In [ ]:
scores_w = np.array([score_w, alternative_w, stable_w])
names_w = ["baseline", "flexible", "stabilized"]
winner_w = int(np.argmin(scores_w))
print("scores:", dict(zip(names_w, np.round(scores_w, 3))))
print("winner:", names_w[winner_w])
assert names_w[winner_w] == "stabilized"

▶ What you'll see: the stabilized score is the minimum of the three candidates.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(names_w, scores_w, color=["gray", "indianred", "seagreen"])
plt.ylabel("decision score"); plt.title("5: final score comparison")
plt.axhline(stable_w, color="seagreen", linestyle="--"); plt.show()

▶ What you'll see: the stabilized bar is lowest, so it is the setting to carry forward in this toy case.

*Why it's done this way:* Variance reduction often matters more than squeezing out a tiny extra training gain. The stabilized score encodes that preference by rewarding settings whose performance is less sensitive to the particular sample drawn.

### 6. Validation estimates the future-facing part of the tradeoff

The tradeoff is ultimately about unseen data. A small validation set lets us see the classic pattern: low-degree models underfit with high bias, high-degree models can overfit with high variance, and a middle setting often wins out-of-sample.

In [ ]:
x_w = np.linspace(-1, 1, 9)
y_w = 1.0 + 2.0 * x_w - 1.5 * x_w**2 + 0.15 * np.sin(9 * x_w)  # toy signal with gentle wiggle.
train_idx_w = np.array([0, 1, 3, 5, 7, 8])
val_idx_w = np.array([2, 4, 6])
print("train points:", len(train_idx_w), "validation points:", len(val_idx_w))

▶ What you'll see: six training points and three validation points for a small model-selection demo.

In [ ]:
degrees_w = np.array([1, 2, 5])
train_mse_w, val_mse_w = [], []
for d_w in degrees_w:
    coef_w = np.polyfit(x_w[train_idx_w], y_w[train_idx_w], d_w)
    pred_train_w = np.polyval(coef_w, x_w[train_idx_w])
    pred_val_w = np.polyval(coef_w, x_w[val_idx_w])
    train_mse_w.append(float(np.mean((y_w[train_idx_w] - pred_train_w) ** 2)))
    val_mse_w.append(float(np.mean((y_w[val_idx_w] - pred_val_w) ** 2)))
print("train MSE:", np.round(train_mse_w, 3))
print("validation MSE:", np.round(val_mse_w, 3))
assert int(degrees_w[np.argmin(val_mse_w)]) == 2

▶ What you'll see: the degree-2 model wins validation even though degree 5 can chase the training points more aggressively.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(degrees_w, train_mse_w, marker="o", label="train MSE")
plt.plot(degrees_w, val_mse_w, marker="o", label="validation MSE")
plt.title("6: validation exposes the tradeoff"); plt.xlabel("polynomial degree"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: training error rewards flexibility, while validation identifies the middle setting as safer.

*Why it's done this way:* Validation is a controlled proxy for future data. It does not prove the selected model is perfect, but it prevents us from using the same examples both to fit a flexible rule and to declare that flexibility successful.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, simulations, polynomial fits, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for the small diagnostic plots in every section.
np.random.seed(0) # make all examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Average three training losses

**Goal.** Compute empirical risk from the lesson's three losses, because every later score starts from an average over examples. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.213, 0.148, 0.420]) # store the three verified toy losses.
print("losses_b1:", losses_b1) # inspect individual losses before summarizing.

▶ What you'll see: three small losses on the same scale.

In [ ]:
risk_b1 = float(losses_b1.mean()) # average the losses to get empirical risk.
print("R_S:", round(risk_b1, 3)) # inspect the raw training score.
assert round(risk_b1, 3) == 0.260 # verify the lesson average.
plt.figure(figsize=(4, 3)); plt.bar(["1", "2", "3"], losses_b1, color="steelblue"); plt.axhline(risk_b1, color="red", linestyle="--"); plt.title("Basic 1: loss average"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the dashed mean line sits between the small and large example losses.

👀 Takeaway: empirical risk is the average training loss, not a single cherry-picked example.

### Basic 2 — Add the complexity cost

**Goal.** Add the method cost to the raw risk, because model selection should compare the full decision score. We build it in 2 steps.

In [ ]:
risk_b2 = 0.260 # use the rounded empirical risk from the lesson arithmetic.
cost_b2 = 0.080 # use the lesson's complexity or operational cost.
print("risk:", risk_b2, "cost:", cost_b2) # inspect the two score components.

▶ What you'll see: the cost is smaller than the risk but large enough to matter.

In [ ]:
score_b2 = risk_b2 + cost_b2 # combine fit and cost.
print("score:", round(score_b2, 3)) # inspect the score used for selection.
assert round(score_b2, 3) == 0.340 # verify the lesson number.
plt.figure(figsize=(4, 3)); plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["steelblue", "orange", "green"]); plt.title("Basic 2: risk plus cost"); plt.show()

▶ What you'll see: the final score is the stacked consequence of fit plus guardrail.

👀 Takeaway: a model with a low raw loss can still lose after its cost is included.

### Basic 3 — Compare two full scores

**Goal.** Compare the baseline score to a flexible alternative, because lower full score is the selection rule in this toy lesson. We build it in 2 steps.

In [ ]:
baseline_b3 = 0.340 # baseline full score.
flexible_b3 = 0.388 # more flexible alternative full score.
print("baseline:", baseline_b3, "flexible:", flexible_b3) # inspect candidates on the same scale.

▶ What you'll see: both candidates are already full decision scores.

In [ ]:
winner_b3 = "baseline" if baseline_b3 < flexible_b3 else "flexible" # choose the lower score.
gap_b3 = flexible_b3 - baseline_b3 # compute the absolute advantage of the baseline.
print("winner:", winner_b3, "gap:", round(gap_b3, 3))
assert winner_b3 == "baseline" and round(gap_b3, 3) == 0.048
plt.figure(figsize=(4, 3)); plt.bar(["baseline", "flexible"], [baseline_b3, flexible_b3], color=["green", "red"]); plt.ylabel("lower is better"); plt.title("Basic 3: same-scale comparison"); plt.show()

▶ What you'll see: the baseline bar is lower by 0.048.

👀 Takeaway: compare full scores, not isolated pieces of different formulas.

### Basic 4 — Convert an absolute gap into a relative gap

**Goal.** Divide the gap by the alternative score, because relative scale tells whether a win is small or substantial. We build it in 2 steps.

In [ ]:
gap_b4 = 0.388 - 0.340 # absolute score difference.
relative_gap_b4 = gap_b4 / 0.388 # normalize by the alternative's scale.
print("gap:", round(gap_b4, 3)) # inspect the raw difference.

▶ What you'll see: the absolute difference is 0.048 score units.

In [ ]:
print("relative gap:", round(relative_gap_b4, 3)) # inspect scale-aware advantage.
assert round(relative_gap_b4, 3) == 0.124 # verify lesson number.
plt.figure(figsize=(4, 3)); plt.bar(["absolute", "relative"], [gap_b4, relative_gap_b4], color=["purple", "teal"]); plt.title("Basic 4: gap readings"); plt.show()

▶ What you'll see: the relative gap is about 12.4%.

👀 Takeaway: relative gaps help judge whether a numerical win is large enough to trust.

### Basic 5 — Apply a stabilization multiplier

**Goal.** Reduce the score by 20%, because stabilization is a variance-control move in the lesson arithmetic. We build it in 2 steps.

In [ ]:
score_b5 = 0.340 # starting full score.
multiplier_b5 = 0.80 # retain 80%, equivalent to a 20% reduction.
print("score before:", score_b5, "multiplier:", multiplier_b5) # inspect inputs.

▶ What you'll see: stabilization uses a simple multiplicative knob.

In [ ]:
stable_b5 = score_b5 * multiplier_b5 # compute stabilized score.
print("stabilized:", round(stable_b5, 3))
assert round(stable_b5, 3) == 0.272
plt.figure(figsize=(4, 3)); plt.bar(["before", "after"], [score_b5, stable_b5], color=["gray", "green"]); plt.title("Basic 5: stabilization lowers score"); plt.show()

▶ What you'll see: the stabilized bar is lower than the original.

👀 Takeaway: stabilizing constraints can improve future-facing scores by reducing brittle variation.

### Basic 6 — Pick the minimum score

**Goal.** Select among baseline, flexible, and stabilized candidates, because the final decision is an argmin over full scores. We build it in 2 steps.

In [ ]:
names_b6 = np.array(["baseline", "flexible", "stabilized"]) # candidate labels.
scores_b6 = np.array([0.340, 0.388, 0.272]) # full decision scores.
print(dict(zip(names_b6, scores_b6))) # inspect the score table.

▶ What you'll see: three candidates on the same score scale.

In [ ]:
best_b6 = int(np.argmin(scores_b6)) # find the lowest score.
print("carry forward:", names_b6[best_b6], "score:", scores_b6[best_b6])
assert names_b6[best_b6] == "stabilized" and round(float(scores_b6[best_b6]), 3) == 0.272
plt.figure(figsize=(4, 3)); plt.bar(names_b6, scores_b6, color=["gray", "red", "green"]); plt.title("Basic 6: final argmin"); plt.ylabel("score"); plt.show()

▶ What you'll see: the stabilized candidate is visibly lowest.

👀 Takeaway: selection is a rule-driven minimum, not a preference for the fanciest model.

### Basic 7 — Compute squared bias

**Goal.** Measure systematic error of the average prediction, because bias is about where predictions center across samples. We build it in 2 steps.

In [ ]:
truth_b7 = 3.0 # true target value at one x.
preds_b7 = np.array([2.0, 2.1, 2.2, 2.1]) # predictions from repeated training samples.
mean_pred_b7 = float(preds_b7.mean()) # average prediction.
print("mean prediction:", round(mean_pred_b7, 3))

▶ What you'll see: the model consistently predicts around 2.1 for a truth of 3.0.

In [ ]:
bias2_b7 = (mean_pred_b7 - truth_b7) ** 2 # squared bias.
print("bias^2:", round(bias2_b7, 3))
assert round(bias2_b7, 3) == 0.810
plt.figure(figsize=(4, 3)); plt.scatter(np.arange(len(preds_b7)), preds_b7, color="teal"); plt.axhline(truth_b7, color="black", linestyle="--", label="truth"); plt.axhline(mean_pred_b7, color="red", label="mean pred"); plt.title("Basic 7: systematic miss"); plt.legend(); plt.show()

▶ What you'll see: the prediction cloud is tightly below the truth, which is high bias.

👀 Takeaway: bias is the squared distance between the average prediction and the target.

### Basic 8 — Compute variance of predictions

**Goal.** Measure sensitivity to the training sample, because variance is prediction spread around the model's own average. We build it in 2 steps.

In [ ]:
preds_b8 = np.array([2.2, 2.6, 3.1, 3.5, 3.6]) # repeated-sample predictions.
center_b8 = float(preds_b8.mean()) # prediction center.
print("center:", round(center_b8, 3))

▶ What you'll see: the average prediction is exactly 3.0 for this toy set.

In [ ]:
variance_b8 = float(np.mean((preds_b8 - center_b8) ** 2)) # average squared spread.
print("variance:", round(variance_b8, 3))
assert round(variance_b8, 3) == 0.284
plt.figure(figsize=(4, 3)); plt.scatter(np.arange(len(preds_b8)), preds_b8, color="purple"); plt.axhline(center_b8, color="black", linestyle="--"); plt.title("Basic 8: prediction spread"); plt.show()

▶ What you'll see: predictions wobble above and below their center.

👀 Takeaway: variance is high when retraining on new samples changes predictions a lot.

### Basic 9 — Add irreducible noise

**Goal.** Add noise to bias² and variance, because some target randomness remains after the best modeling choice. We build it in 2 steps.

In [ ]:
bias2_b9 = 0.000 # no systematic miss in this toy decomposition.
variance_b9 = 0.284 # sample sensitivity from Basic 8.
noise_b9 = 0.090 # irreducible sigma squared.
print("pieces:", bias2_b9, variance_b9, noise_b9)

▶ What you'll see: the removable and non-removable pieces are listed separately.

In [ ]:
expected_error_b9 = bias2_b9 + variance_b9 + noise_b9 # decomposition total.
print("expected squared error:", round(expected_error_b9, 3))
assert round(expected_error_b9, 3) == 0.374
plt.figure(figsize=(4, 3)); plt.bar(["bias²", "variance", "noise"], [bias2_b9, variance_b9, noise_b9], color=["gray", "purple", "orange"]); plt.title("Basic 9: decomposition sum"); plt.show()

▶ What you'll see: even with zero bias, variance and noise keep error above zero.

👀 Takeaway: generalization error has parts you can tune down and a noise floor you cannot erase.

### Basic 10 — Plot underfit, balanced, and overfit curves

**Goal.** Visualize model flexibility, because bias and variance are easiest to see as curves that are too flat, just right, or too wiggly. We build it in 2 steps.

In [ ]:
x_b10 = np.linspace(-1, 1, 9) # small input grid.
y_b10 = 1 + 2 * x_b10 - 1.5 * x_b10**2 + 0.15 * np.sin(9 * x_b10) # deterministic toy observations.
degrees_b10 = [1, 2, 5] # underfit, balanced, and flexible polynomial degrees.
print("degrees:", degrees_b10)

▶ What you'll see: three degrees that represent increasing flexibility.

In [ ]:
xx_b10 = np.linspace(-1, 1, 200) # smooth grid for plotting fitted curves.
plt.figure(figsize=(5, 3))
plt.scatter(x_b10, y_b10, color="black", label="data")
for deg_b10 in degrees_b10:
    coef_b10 = np.polyfit(x_b10, y_b10, deg_b10)
    plt.plot(xx_b10, np.polyval(coef_b10, xx_b10), label=f"degree {deg_b10}")
assert len(degrees_b10) == 3
plt.title("Basic 10: flexibility changes curve shape"); plt.legend(); plt.show()

▶ What you'll see: degree 1 is too rigid, degree 2 follows the main pattern, and degree 5 bends more sharply.

👀 Takeaway: flexibility is the knob that usually lowers bias while raising variance risk.

## 🟡 Easy

### Easy 1 — Simulate bias and variance for two estimators

**Goal.** Compare a constant estimator with a noisy flexible estimator, because the tradeoff is a choice between systematic miss and sample sensitivity. We build it in 3 steps.

In [ ]:
truth_e1 = 3.0 # target at one input.
constant_preds_e1 = np.full(8, 2.4) # stable but systematically low predictions.
flex_preds_e1 = np.array([2.1, 2.6, 3.0, 3.4, 3.9, 2.7, 3.5, 3.8]) # less biased but more variable predictions.
print("constant mean:", constant_preds_e1.mean(), "flex mean:", round(float(flex_preds_e1.mean()), 3))

▶ What you'll see: the flexible estimator centers closer to the truth but moves around more.

In [ ]:
bias2_e1 = np.array([(constant_preds_e1.mean() - truth_e1) ** 2, (flex_preds_e1.mean() - truth_e1) ** 2])
var_e1 = np.array([np.mean((constant_preds_e1 - constant_preds_e1.mean()) ** 2), np.mean((flex_preds_e1 - flex_preds_e1.mean()) ** 2)])
print("bias^2:", np.round(bias2_e1, 3), "variance:", np.round(var_e1, 3))
assert round(float(bias2_e1[0]), 3) == 0.360 and round(float(var_e1[0]), 3) == 0.000

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["const bias²", "const var", "flex bias²", "flex var"], [bias2_e1[0], var_e1[0], bias2_e1[1], var_e1[1]], color=["gray", "gray", "teal", "teal"]); plt.xticks(rotation=20); plt.title("Easy 1: estimator tradeoff pieces"); plt.show()

▶ What you'll see: one estimator pays in bias, the other pays in variance.

👀 Takeaway: bias and variance are separate costs, so improving one can worsen the other.

### Easy 2 — Use validation to choose polynomial degree

**Goal.** Fit polynomial degrees on training data and score validation data, because validation estimates which flexibility level survives unseen examples. We build it in 3 steps.

In [ ]:
x_e2 = np.linspace(-1, 1, 9)
y_e2 = 1.0 + 2.0 * x_e2 - 1.5 * x_e2**2 + 0.15 * np.sin(9 * x_e2)
train_e2 = np.array([0, 1, 3, 5, 7, 8]); val_e2 = np.array([2, 4, 6])
print("train/val sizes:", len(train_e2), len(val_e2))

▶ What you'll see: a deliberately tiny train/validation split.

In [ ]:
degrees_e2 = np.array([1, 2, 5])
val_mse_e2 = []
for degree_e2 in degrees_e2:
    coef_e2 = np.polyfit(x_e2[train_e2], y_e2[train_e2], int(degree_e2))
    pred_e2 = np.polyval(coef_e2, x_e2[val_e2])
    val_mse_e2.append(float(np.mean((y_e2[val_e2] - pred_e2) ** 2)))
print("validation MSE:", np.round(val_mse_e2, 4))
assert int(degrees_e2[np.argmin(val_mse_e2)]) == 2

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(degrees_e2, val_mse_e2, marker="o", color="seagreen"); plt.xticks(degrees_e2); plt.xlabel("degree"); plt.ylabel("validation MSE"); plt.title("Easy 2: validation selects degree 2"); plt.show()

▶ What you'll see: the validation curve is lowest at the middle degree.

👀 Takeaway: the best training fit is not automatically the best validation fit.

### Easy 3 — Bootstrap prediction variance

**Goal.** Refit a line on bootstrap samples and watch one prediction vary, because variance is sensitivity to which sample we happened to draw. We build it in 3 steps.

In [ ]:
x_e3 = np.linspace(0, 1, 8)
y_e3 = 1 + 2 * x_e3 + np.array([0.05, -0.02, 0.03, 0.08, -0.04, 0.02, -0.01, 0.04])
rng_e3 = np.random.default_rng(3)
print("data points:", len(x_e3))

▶ What you'll see: eight nearly linear observations.

In [ ]:
preds_e3 = []
for boot_e3 in range(40):
    idx_e3 = rng_e3.integers(0, len(x_e3), len(x_e3))
    coef_e3 = np.polyfit(x_e3[idx_e3], y_e3[idx_e3], 1)
    preds_e3.append(float(np.polyval(coef_e3, 0.8)))
preds_e3 = np.array(preds_e3)
print("prediction mean:", round(float(preds_e3.mean()), 3), "variance:", round(float(preds_e3.var()), 5))
assert preds_e3.var() > 0

In [ ]:
plt.figure(figsize=(5, 3)); plt.hist(preds_e3, bins=10, color="purple", edgecolor="white"); plt.axvline(preds_e3.mean(), color="black", linestyle="--"); plt.title("Easy 3: bootstrap prediction spread"); plt.xlabel("prediction at x=0.8"); plt.show()

▶ What you'll see: repeated fits do not give exactly the same prediction.

👀 Takeaway: variance is visible as a distribution of predictions over plausible training samples.

### Easy 4 — Add an L2-style penalty to model selection

**Goal.** Penalize larger coefficients, because flexible curves can hide their instability behind a low training MSE. We build it in 3 steps.

In [ ]:
x_e4 = np.linspace(-1, 1, 9)
y_e4 = 1 + 2 * x_e4 - 1.5 * x_e4**2 + 0.15 * np.sin(9 * x_e4)
degrees_e4 = np.array([1, 2, 5])
lam_e4 = 0.01
print("lambda:", lam_e4)

▶ What you'll see: a small penalty strength for coefficient size.

In [ ]:
scores_e4 = []
for degree_e4 in degrees_e4:
    coef_e4 = np.polyfit(x_e4, y_e4, int(degree_e4))
    mse_e4 = float(np.mean((y_e4 - np.polyval(coef_e4, x_e4)) ** 2))
    penalty_e4 = lam_e4 * float(np.sum(coef_e4 ** 2))
    scores_e4.append(mse_e4 + penalty_e4)
print("penalized scores:", np.round(scores_e4, 4))
assert len(scores_e4) == 3 and min(scores_e4) >= 0

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(degrees_e4, scores_e4, marker="o", color="darkorange"); plt.xticks(degrees_e4); plt.xlabel("degree"); plt.ylabel("MSE + λ||coef||²"); plt.title("Easy 4: penalized selection score"); plt.show()

▶ What you'll see: the score includes both raw fit and a cost for large coefficients.

👀 Takeaway: regularization turns complexity into an explicit part of the selection arithmetic.

### Easy 5 — Estimate noise with repeated observations

**Goal.** Measure irreducible noise from repeated labels at the same input, because no model can predict random label jitter perfectly. We build it in 3 steps.

In [ ]:
repeats_e5 = np.array([3.1, 2.9, 3.0, 3.2, 2.8]) # repeated measurements at the same x.
mean_e5 = float(repeats_e5.mean()) # best constant prediction for that x.
print("repeat mean:", round(mean_e5, 3))

▶ What you'll see: the repeated observations center at 3.0.

In [ ]:
noise_e5 = float(np.mean((repeats_e5 - mean_e5) ** 2)) # residual variance around the repeat mean.
print("estimated noise variance:", round(noise_e5, 3))
assert round(noise_e5, 3) == 0.020

In [ ]:
plt.figure(figsize=(4, 3)); plt.scatter(np.zeros_like(repeats_e5), repeats_e5, color="teal"); plt.axhline(mean_e5, color="black", linestyle="--"); plt.title("Easy 5: repeated-label noise"); plt.xticks([]); plt.ylabel("observed y"); plt.show()

▶ What you'll see: points at the same x still differ vertically.

👀 Takeaway: irreducible noise is variation in the target that remains even when the input is fixed.

## 🔴 Advanced

### Advanced 1 — Decompose error across many simulated training sets

**Goal.** Fit many models on noisy samples and decompose one test-point error, because this is the operational meaning of the bias–variance formula. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11)
x0_a1 = 0.4
true_a1 = 1 + 2 * x0_a1 - 1.5 * x0_a1**2
preds_a1 = []
print("true f(x0):", round(true_a1, 3))

▶ What you'll see: the fixed test point has a known noiseless target.

In [ ]:
for rep_a1 in range(80):
    x_train_a1 = rng_a1.uniform(-1, 1, 12)
    y_train_a1 = 1 + 2 * x_train_a1 - 1.5 * x_train_a1**2 + rng_a1.normal(0, 0.15, 12)
    coef_a1 = np.polyfit(x_train_a1, y_train_a1, 2)
    preds_a1.append(float(np.polyval(coef_a1, x0_a1)))
preds_a1 = np.array(preds_a1)
print("mean prediction:", round(float(preds_a1.mean()), 3))

In [ ]:
bias2_a1 = (float(preds_a1.mean()) - true_a1) ** 2
var_a1 = float(np.mean((preds_a1 - preds_a1.mean()) ** 2))
noise_a1 = 0.15 ** 2
total_a1 = bias2_a1 + var_a1 + noise_a1
print("bias², variance, noise:", round(bias2_a1, 4), round(var_a1, 4), round(noise_a1, 4))
assert var_a1 > 0 and round(noise_a1, 4) == 0.0225

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["bias²", "variance", "noise", "sum"], [bias2_a1, var_a1, noise_a1, total_a1], color=["gray", "purple", "orange", "green"]); plt.title("Advanced 1: simulated decomposition"); plt.show()

▶ What you'll see: one bar for each source of expected squared error.

👀 Takeaway: the formula becomes measurable when we imagine retraining on many possible samples.

### Advanced 2 — Sweep degree for train and validation error

**Goal.** Trace underfitting and overfitting across a wider degree grid, because the U-shape is the visual signature of the tradeoff. We build it in 4 steps.

In [ ]:
x_a2 = np.linspace(-1, 1, 14)
y_a2 = 1 + 2 * x_a2 - 1.5 * x_a2**2 + 0.2 * np.sin(8 * x_a2)
train_a2 = np.arange(0, 14, 2); val_a2 = np.arange(1, 14, 2)
degrees_a2 = np.arange(1, 7)
print("degrees:", degrees_a2)

▶ What you'll see: six candidate polynomial degrees.

In [ ]:
train_err_a2, val_err_a2 = [], []
for degree_a2 in degrees_a2:
    coef_a2 = np.polyfit(x_a2[train_a2], y_a2[train_a2], int(degree_a2))
    train_err_a2.append(float(np.mean((y_a2[train_a2] - np.polyval(coef_a2, x_a2[train_a2])) ** 2)))
    val_err_a2.append(float(np.mean((y_a2[val_a2] - np.polyval(coef_a2, x_a2[val_a2])) ** 2)))
print("train errors:", np.round(train_err_a2, 4))
print("val errors:", np.round(val_err_a2, 4))

In [ ]:
best_degree_a2 = int(degrees_a2[np.argmin(val_err_a2)])
print("best validation degree:", best_degree_a2)
assert best_degree_a2 in degrees_a2

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(degrees_a2, train_err_a2, marker="o", label="train"); plt.plot(degrees_a2, val_err_a2, marker="o", label="validation"); plt.axvline(best_degree_a2, color="red", linestyle="--"); plt.xlabel("degree"); plt.ylabel("MSE"); plt.title("Advanced 2: bias-variance degree sweep"); plt.legend(); plt.show()

▶ What you'll see: training error tends to fall with degree, while validation chooses a finite middle point.

👀 Takeaway: capacity should be tuned by future-facing error, not by training error alone.

### Advanced 3 — Compare bagging-style averaging with one wiggly model

**Goal.** Average several high-variance fits, because averaging reduces variance when model errors are not perfectly identical. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(13)
x_a3 = np.linspace(-1, 1, 12)
y_a3 = 1 + 2 * x_a3 - 1.5 * x_a3**2 + rng_a3.normal(0, 0.12, len(x_a3))
xx_a3 = np.linspace(-1, 1, 100)
print("training points:", len(x_a3))

▶ What you'll see: a small noisy dataset where flexible curves can wobble.

In [ ]:
curves_a3 = []
for rep_a3 in range(25):
    idx_a3 = rng_a3.choice(len(x_a3), size=8, replace=False)
    coef_a3 = np.polyfit(x_a3[idx_a3], y_a3[idx_a3], 5)
    curves_a3.append(np.polyval(coef_a3, xx_a3))
curves_a3 = np.array(curves_a3)
avg_curve_a3 = curves_a3.mean(axis=0)
print("curve matrix shape:", curves_a3.shape)

In [ ]:
spread_one_point_a3 = float(curves_a3[:, 70].var())
spread_average_a3 = float(np.var(avg_curve_a3))
print("bootstrap spread at one x:", round(spread_one_point_a3, 4))
assert spread_one_point_a3 > 0

In [ ]:
plt.figure(figsize=(5, 3))
for k_a3 in range(8):
    plt.plot(xx_a3, curves_a3[k_a3], color="gray", alpha=0.35)
plt.plot(xx_a3, avg_curve_a3, color="crimson", linewidth=2, label="average curve")
plt.scatter(x_a3, y_a3, color="black", s=18); plt.title("Advanced 3: averaging reduces wiggle"); plt.legend(); plt.show()

▶ What you'll see: individual gray subset curves wiggle, while the red average is smoother.

👀 Takeaway: averaging is a variance-reduction strategy, which is why ensembles often generalize well.

### Advanced 4 — Cross-validation reduces split luck

**Goal.** Average validation error across folds, because one validation split can be noisy when the dataset is small. We build it in 4 steps.

In [ ]:
x_a4 = np.linspace(-1, 1, 12)
y_a4 = 1 + 2 * x_a4 - 1.5 * x_a4**2 + 0.12 * np.sin(10 * x_a4)
fold_ids_a4 = np.arange(len(x_a4)) % 3
candidate_degrees_a4 = np.array([1, 2, 5])
print("fold counts:", [int(np.sum(fold_ids_a4 == f_a4)) for f_a4 in range(3)])

▶ What you'll see: three folds with four validation points each.

In [ ]:
cv_errors_a4 = []
for degree_a4 in candidate_degrees_a4:
    fold_errors_a4 = []
    for fold_a4 in range(3):
        val_mask_a4 = fold_ids_a4 == fold_a4
        train_mask_a4 = ~val_mask_a4
        coef_a4 = np.polyfit(x_a4[train_mask_a4], y_a4[train_mask_a4], int(degree_a4))
        pred_a4 = np.polyval(coef_a4, x_a4[val_mask_a4])
        fold_errors_a4.append(float(np.mean((y_a4[val_mask_a4] - pred_a4) ** 2)))
    cv_errors_a4.append(float(np.mean(fold_errors_a4)))
print("CV errors:", np.round(cv_errors_a4, 4))

In [ ]:
best_degree_a4 = int(candidate_degrees_a4[np.argmin(cv_errors_a4)])
print("best CV degree:", best_degree_a4)
assert best_degree_a4 in candidate_degrees_a4

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar([str(d_a4) for d_a4 in candidate_degrees_a4], cv_errors_a4, color="teal"); plt.xlabel("degree"); plt.ylabel("mean fold MSE"); plt.title("Advanced 4: cross-validation averages split noise"); plt.show()

▶ What you'll see: each bar is an average over three validation folds, not one lucky split.

👀 Takeaway: cross-validation makes the model-choice score less sensitive to a single partition.

### Advanced 5 — Tune regularization as a variance knob

**Goal.** Sweep ridge-style polynomial regularization, because λ directly controls the bias-variance bargain by shrinking coefficients. We build it in 5 steps.

In [ ]:
x_a5 = np.linspace(-1, 1, 16)
y_a5 = 1 + 2 * x_a5 - 1.5 * x_a5**2 + 0.25 * np.sin(7 * x_a5)
train_a5 = np.arange(0, 16, 2); val_a5 = np.arange(1, 16, 2)
lams_a5 = np.array([0.0, 0.001, 0.01, 0.1, 1.0])
print("lambda grid:", lams_a5)

▶ What you'll see: regularization strengths from none to strong.

In [ ]:
X_train_a5 = np.vstack([x_a5[train_a5] ** p_a5 for p_a5 in range(6)]).T
X_val_a5 = np.vstack([x_a5[val_a5] ** p_a5 for p_a5 in range(6)]).T
print("design shape:", X_train_a5.shape)

In [ ]:
val_scores_a5 = []
coefs_a5 = []
I_a5 = np.eye(X_train_a5.shape[1]); I_a5[0, 0] = 0.0
for lam_a5 in lams_a5:
    coef_a5 = np.linalg.solve(X_train_a5.T @ X_train_a5 + lam_a5 * I_a5, X_train_a5.T @ y_a5[train_a5])
    pred_val_a5 = X_val_a5 @ coef_a5
    val_scores_a5.append(float(np.mean((y_a5[val_a5] - pred_val_a5) ** 2)))
    coefs_a5.append(coef_a5)
print("validation scores:", np.round(val_scores_a5, 4))
assert len(val_scores_a5) == len(lams_a5)

In [ ]:
best_idx_a5 = int(np.argmin(val_scores_a5))
best_lam_a5 = float(lams_a5[best_idx_a5])
coef_norms_a5 = np.array([np.linalg.norm(c_a5[1:]) for c_a5 in coefs_a5])
print("best lambda:", best_lam_a5, "coef norms:", np.round(coef_norms_a5, 3))
assert best_lam_a5 in lams_a5

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(lams_a5, val_scores_a5, marker="o", label="validation MSE"); plt.xscale("symlog", linthresh=0.001); plt.xlabel("λ"); plt.ylabel("MSE"); plt.title("Advanced 5: regularization sweep"); plt.legend(); plt.show()

▶ What you'll see: validation error changes as λ shrinks the polynomial coefficients.

👀 Takeaway: λ is a tunable stability knob; choose it with validation evidence rather than training loss alone.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Expected error separates into systematic miss, sensitivity to samples, and irreducible noise.

The bias–variance tradeoff uses empirical risk, validation behavior, and a cost-aware decision score. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))
    return rungs

def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)

def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ \mathbb E[(Y-\hat f(x))^2]=\text{bias}^2+\text{variance}+\sigma^2 $$. The next cell recomputes the exact loss average, cost, gap, and stabilized score from the plan.

In [ ]:

def the_bias_variance_tradeoff_method():
    losses = np.array([0.213, 0.148, 0.42], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.080
    score = round(empirical_risk + cost, 3)
    alternative = 0.388
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = the_bias_variance_tradeoff_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.781)
assert np.isclose(round(lesson_check["risk"], 3), 0.260)
assert np.isclose(round(lesson_check["score"], 3), 0.340)
assert np.isclose(round(lesson_check["gap"], 3), 0.048)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.124)
assert np.isclose(round(lesson_check["stable"], 3), 0.272)


The assertions above keep the notebook and lesson prose on the same algorithmic scale.

In [ ]:

def safe_stratify(y):
    values, counts = np.unique(y, return_counts=True)
    if len(values) < 2:
        return None
    if counts.min() < 2:
        return None
    return y

def plot_2d_projection(ax, X, y, title):
    x_plot = X[:, :2]
    ax.scatter(x_plot[:, 0], x_plot[:, 1], c=y, cmap="viridis", s=16, alpha=0.75)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

def logistic_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    candidates = []
    for c_value in [0.05, 0.2, 1.0, 5.0]:
        model = LogisticRegression(C=c_value, max_iter=2000)
        model.fit(x_tr_s, y_tr)
        tr_prob = model.predict_proba(x_tr_s)
        te_prob = model.predict_proba(x_te_s)
        tr_pred = model.predict(x_tr_s)
        te_pred = model.predict(x_te_s)
        labels = model.classes_
        train_loss = log_loss(y_tr, tr_prob, labels=labels)
        val_loss = log_loss(y_te, te_prob, labels=labels)
        cost = 0.02 / c_value
        candidates.append({
            "C": c_value,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "gap": float(val_loss - train_loss),
            "accuracy": float(accuracy_score(y_te, te_pred)),
            "cost": float(cost),
            "score": float(val_loss + cost),
            "pred": te_pred,
        })
    raw_winner = min(candidates, key=lambda item: item["val_loss"])
    fixed_winner = min(candidates, key=lambda item: item["score"])
    return candidates, raw_winner, fixed_winner

def run_logistic_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(X, y)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": fixed_winner["val_loss"],
            "gap": fixed_winner["gap"],
            "accuracy": fixed_winner["accuracy"],
            "C": fixed_winner["C"],
            "score": fixed_winner["score"],
            "raw_C": raw_winner["C"],
        })
    return rows

def bias_variance_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    models = [
        ("linear", make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))),
        ("flexible-knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ]
    rows = []
    for label, model in models:
        model.fit(x_tr, y_tr)
        train_error = 1.0 - accuracy_score(y_tr, model.predict(x_tr))
        val_error = 1.0 - accuracy_score(y_te, model.predict(x_te))
        variance_proxy = abs(val_error - train_error)
        complexity_cost = 0.01 if label == "linear" else 0.04
        rows.append({
            "label": label,
            "train_error": float(train_error),
            "val_error": float(val_error),
            "variance_proxy": float(variance_proxy),
            "score": float(val_error + variance_proxy + complexity_cost),
        })
    return rows

def run_bias_variance_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates = bias_variance_candidates_for_rung(X, y)
        winner = min(candidates, key=lambda item: item["score"])
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": winner["val_error"],
            "gap": winner["variance_proxy"],
            "model": winner["label"],
            "score": winner["score"],
        })
    return rows

def add_intercept(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])

def train_binary_perceptron(X, y_signed, epochs=60):
    X_aug = add_intercept(X)
    weights = np.zeros(X_aug.shape[1])
    mistakes = []
    for epoch in range(epochs):
        errors = 0
        for xi, yi in zip(X_aug, y_signed):
            margin = yi * float(np.dot(weights, xi))
            if margin <= 0:
                weights = weights + yi * xi
                errors = errors + 1
        mistakes.append(errors)
        if errors == 0:
            break
    return weights, mistakes

def train_ovr_perceptron(X, y, epochs=60):
    classes = np.unique(y)
    weights = []
    histories = []
    for cls in classes:
        y_signed = np.where(y == cls, 1, -1)
        w, hist = train_binary_perceptron(X, y_signed, epochs=epochs)
        weights.append(w)
        histories.append(hist)
    return classes, np.vstack(weights), histories

def predict_ovr_perceptron(classes, weights, X):
    scores = add_intercept(X).dot(weights.T)
    return classes[np.argmax(scores, axis=1)]

def perceptron_predictor(x_tr, y_tr, x_te):
    classes, weights, histories = train_ovr_perceptron(x_tr, y_tr, epochs=60)
    return predict_ovr_perceptron(classes, weights, x_te)

def run_perceptron_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        stratify = safe_stratify(y)
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        classes, weights, histories = train_ovr_perceptron(x_tr_s, y_tr, epochs=60)
        pred = predict_ovr_perceptron(classes, weights, x_te_s)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": float(accuracy_score(y_te, pred)),
            "history": histories,
        })
    return rows


## The dataset ladder

D1 is inspectable by hand; D5 is a real 30-dimensional breast-cancer classification problem.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    values, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  class counts:", dict(zip(values.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


## Run the same method across D1–D5

The metric follows the plan: validation loss and generalization gap for 3.1–3.3, accuracy for 3.4.

In [ ]:

results = run_bias_variance_ladder()
print("rung | validation_error | variance_proxy | chosen_model | score")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['gap']:.3f} | {row['model']} | {row['score']:.3f}")


## Results visualization

The closing figure has one panel per rung plus a summary curve over D1–D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, (name, X, y), row in zip(flat_axes[:5], rungs, results):
    plot_2d_projection(ax, X, y, f"D{row['rung']} validation loss={row['metric']:.2f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o", label="validation loss")
if "bias" != "perceptron":
    flat_axes[5].plot([row["rung"] for row in results], [abs(row["gap"]) for row in results], marker="s", label="gap")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("loss / gap")
flat_axes[5].legend()
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

The hardest rung demonstrates why the raw term alone is not the decision rule.

In [ ]:

d5_name, d5_X, d5_y = rungs[-1]
d5_candidates = bias_variance_candidates_for_rung(d5_X, d5_y)
raw_winner = min(d5_candidates, key=lambda item: item["val_error"])
fixed_winner = min(d5_candidates, key=lambda item: item["score"])
print("D5:", d5_name)
print("wrong raw winner:", raw_winner["label"], "val error", round(raw_winner["val_error"], 3))
print("fixed cost/gap-aware winner:", fixed_winner["label"], "score", round(fixed_winner["score"], 3))
print("fixed variance proxy:", round(fixed_winner["variance_proxy"], 3))
assert fixed_winner["score"] <= raw_winner["val_error"] + raw_winner["variance_proxy"] + 0.04 + 1e-9


## Evaluate it + Practice

- Compare the metric with a no-skill baseline or `logistic_baseline`.
- Sanity check: shuffle labels and confirm the score degrades.
- Ablation: turn off the cost or scaling fix and watch the D5 choice or metric change.
- Failure signals: a large validation gap, a scale mismatch, or a raw-only winner.

Practice 1: Change one cost and rerun the D5 selection.

Practice 2: Repeat the D5 split with a different seed and compare the gap.

Practice 3: For skewed classes, add macro-F1 and compare it with accuracy.